# Algorithm Verification: C&CG vs BDCP

Runs both algorithms on the same instances and flags discrepancies.

**One-time setup** — run from the terminal before opening this notebook:
```bash
cd <folder containing setup.py>
pip install -e .
```

In [1]:
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt

from rcflp.instance import instancemaker
from rcflp.nominal  import solve_nominal
from rcflp.ccg      import solve_CCG
from rcflp.bdcp     import solve_BDCP

## 1. Experiment settings

In [2]:
Hn    = 2
Rn    = 3
BIG_M = 10_000

# --- Overnight grid ---
# Covers a range of instance sizes, v, w, and Gamma values.
# Expected runtime: 4-8 hours depending on hardware.
I_J_values          = [(15, 10)]#, (15, 10), (20, 15), (25, 15)]
value_max_scales    = [0.75]#, 1.0]
congestion_costs    = [100]#, 100]
uncertainty_budgets = [1, 2, 3]

OBJ_TOL_REL  = 0.01   # 1% relative gap — C&CG termination
OBJ_TOL_ABS  = 0.01   # absolute gap   — BDCP termination
TIME_MASTER  = 500    # per individual master solve (seconds)

# Per-instance time limits scale with size (set in the run loop below):
# |I|<=10 -> 5 min,  |I|<=20 -> 15 min,  |I|>20 -> 30 min

parameters = list(itertools.product(
    value_max_scales, congestion_costs, uncertainty_budgets, I_J_values))
print(f'{len(parameters)} instances x 2 algorithms = {2*len(parameters)} solves')

3 instances x 2 algorithms = 6 solves


## 2. Run both algorithms

In [3]:
import time
results = []
run_start = time.time()

for idx, (v, w, gamma, (In, Jn)) in enumerate(parameters):
    print(f'\n[{idx+1}/{len(parameters)}] |I|={In} |J|={Jn}  v={v}  w={w}  Gamma={gamma}')
    print(f'  Elapsed so far: {(time.time()-run_start)/3600:.2f}h')

    inst  = instancemaker(In, Jn, Rn, v, w)
    nom   = solve_nominal(inst)
    x_nom = nom['x_jr']
    print(f'  Nominal profit = {nom["profit"]:.1f}')

    # Scale time limit with instance size so total run stays manageable.
    # Small (I<=10): 5 min,  Medium (I<=20): 15 min,  Large (I<=30): 30 min
    if In <= 10:
        time_limit = 300
    elif In <= 20:
        time_limit = 900
    else:
        time_limit = 1800
    print(f'  Time limit: {time_limit//60} min per algorithm')

    ccg = solve_CCG(
        inst, gamma, Hn, x_nom,
        tol=OBJ_TOL_REL, big_M=BIG_M,
        time_limit=time_limit, master_time_limit=TIME_MASTER,
        verbose=True)
    print(f'  C&CG  profit={ccg["profit_LB"]:.1f}  iters={ccg["n_iter"]}  t={ccg["runtime"]:.1f}s  converged={ccg["converged"]}')

    bdcp = solve_BDCP(
        inst, gamma, Hn, x_nom,
        tol=OBJ_TOL_ABS, big_M=BIG_M,
        time_limit=time_limit, master_time_limit=TIME_MASTER,
        verbose=True)
    print(f'  BDCP  profit={bdcp["profit_LB"]:.1f}  iters={bdcp["n_iter"]}  t={bdcp["runtime"]:.1f}s  converged={bdcp["converged"]}')

    denom    = max(abs(ccg['profit_LB']), abs(bdcp['profit_LB']), 1.0)
    abs_diff = abs(ccg['profit_LB'] - bdcp['profit_LB'])
    rel_diff = abs_diff / denom
    mismatch = (rel_diff > 0.01) and (abs_diff > 5.0)
    print(f'  Diff = {abs_diff:.1f} ({100*rel_diff:.2f}%)  ' + ('*** MISMATCH ***' if mismatch else 'OK'))

    results.append({
        'I': In, 'J': Jn, 'v': v, 'w': w, 'Gamma': gamma,
        'nom_profit':   round(nom['profit'],      1),
        'CCG_profit':   round(ccg['profit_LB'],   1),
        'CCG_LB':       round(-ccg['LB'],         1),
        'CCG_UB':       round(-ccg['UB'],         1),
        'CCG_iters':    ccg['n_iter'],
        'CCG_time':     round(ccg['runtime'],     1),
        'CCG_conv':     ccg['converged'],
        'BDCP_profit':  round(bdcp['profit_LB'],  1),
        'BDCP_LB':      round(-bdcp['LB'],        1),
        'BDCP_UB':      round(-bdcp['UB'],        1),
        'BDCP_iters':   bdcp['n_iter'],
        'BDCP_time':    round(bdcp['runtime'],    1),
        'BDCP_conv':    bdcp['converged'],
        'abs_diff':     round(abs_diff,           1),
        'rel_diff_pct': round(100 * rel_diff,     2),
        'mismatch':     mismatch,
        '_ccg_log':     ccg['iter_log'],
        '_bdcp_log':    bdcp['iter_log'],
    })

    # Save intermediate results after every instance in case of interruption
    pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in results]).to_excel('verification_results.xlsx', index=False)

print(f'\n=== ALL DONE — total time: {(time.time()-run_start)/3600:.2f}h ===')


[1/3] |I|=15 |J|=10  v=0.75  w=100  Gamma=1
  Elapsed so far: 0.00h
Set parameter Username
Set parameter LicenseID to value 2785999
Academic license - for non-commercial use only - expires 2027-03-02
  Nominal profit = 173716.7
  Time limit: 15 min per algorithm
  iter   1 | LB=  -154157.60 | UB=  -123588.40 | gap=24.73% | sub=0.1s | master=0.2s
  iter   2 | LB=  -133807.44 | UB=  -123588.40 | gap=8.27% | sub=0.1s | master=1.6s
  iter   3 | LB=  -125519.18 | UB=  -123588.40 | gap=1.56% | sub=0.1s | master=4.2s
  iter   4 | LB=  -125519.18 | UB=  -123588.40 | gap=1.56% | sub=0.1s | master=7.5s
  iter   5 | LB=  -124104.32 | UB=  -123588.40 | gap=0.42% | sub=0.1s | master=12.4s
  C&CG  profit=123588.4  iters=5  t=26.6s  converged=True
  iter   1 | LB=-61951327.98 | UB=  -123588.40 | gap=50027.14% | sub=0.3s | master=0.0s
  iter   2 | LB=-55702339.20 | UB=  -123588.40 | gap=44970.85% | sub=0.2s | master=0.0s
  iter   3 | LB=-31433402.98 | UB=  -123588.40 | gap=25333.94% | sub=0.1s | mast

KeyboardInterrupt: 

## 3. Summary table

In [ ]:
display_cols = [
    'I','J','v','w','Gamma','nom_profit',
    'CCG_profit','CCG_iters','CCG_time','CCG_conv',
    'BDCP_profit','BDCP_iters','BDCP_time','BDCP_conv',
    'abs_diff','rel_diff_pct','mismatch',
]
df = pd.DataFrame([{c: r[c] for c in display_cols} for r in results])

def highlight(row):
    color = 'background-color: #ffcccc' if row['mismatch'] else ''
    return [color] * len(row)

display(df.style.apply(highlight, axis=1).format(precision=1))
print(f'\nMismatches: {df["mismatch"].sum()} / {len(df)}')
print(f'Timed out (not converged): CCG={len(df[~df["CCG_conv"]])}  BDCP={len(df[~df["BDCP_conv"]])}')

## 4. Convergence plots

In [ ]:
# Plots mismatched instances; falls back to first instance if none
to_plot = [r for r in results if r['mismatch']] or results[:1]

for r in to_plot:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"|I|={r['I']} |J|={r['J']} v={r['v']} w={r['w']} Gamma={r['Gamma']}")

    for ax, log, name, color in [
        (axes[0], r['_ccg_log'],  'C&CG', 'steelblue'),
        (axes[1], r['_bdcp_log'], 'BDCP', 'darkorange'),
    ]:
        iters = [d['iter'] for d in log]
        ub    = [-d['UB'] for d in log]
        lb    = [-d['LB'] for d in log]
        ax.plot(iters, ub, label='UB (profit)', color=color, lw=2)
        ax.plot(iters, lb, label='LB (profit)', color=color, lw=2, ls='--')
        ax.set_title(name)
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Profit')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 5. Time breakdown: subproblem vs master

In [ ]:
# For each instance, compute total time spent in subproblem vs master
timing_rows = []
for r in results:
    for algo, log in [('CCG', r['_ccg_log']), ('BDCP', r['_bdcp_log'])]:
        total_sub    = sum(d['t_sub']    for d in log)
        total_master = sum(d['t_master'] for d in log)
        total        = total_sub + total_master
        timing_rows.append({
            'I': r['I'], 'J': r['J'], 'v': r['v'], 'w': r['w'], 'Gamma': r['Gamma'],
            'algo':          algo,
            'n_iter':        len(log),
            't_sub_total':   round(total_sub,    1),
            't_master_total':round(total_master, 1),
            'pct_sub':       round(100 * total_sub    / (total + 1e-9), 1),
            'pct_master':    round(100 * total_master / (total + 1e-9), 1),
        })

tdf = pd.DataFrame(timing_rows)
display(tdf)

# Summary: average % time in subproblem vs master, by algorithm and instance size
print('\nAverage % time in subproblem by algorithm:')
display(tdf.groupby('algo')[['pct_sub','pct_master']].mean().round(1))

## 6. Performance comparison (iterations and time)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, label in [
    (axes[0], ('CCG_iters', 'BDCP_iters'), 'Iterations'),
    (axes[1], ('CCG_time',  'BDCP_time'),  'Time (s)'),
]:
    ax.scatter(df[metric[0]], df[metric[1]], c=df['Gamma'], cmap='viridis', s=60, alpha=0.8)
    lim = max(df[metric[0]].max(), df[metric[1]].max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', lw=1, label='equal')
    ax.set_xlabel(f'C&CG {label}')
    ax.set_ylabel(f'BDCP {label}')
    ax.set_title(f'{label}: C&CG vs BDCP (colour = Gamma)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 7. Per-iteration timing for Gamma=3 CCG

In [ ]:
r = [r for r in results if r['Gamma'] == 3][0]
log = r['_ccg_log']

iters      = [d['iter']     for d in log]
t_master   = [d['t_master'] for d in log]
t_sub      = [d['t_sub']    for d in log]
gap        = [d['gap_pct']  for d in log]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

ax1.bar(iters, t_master, label='master', color='steelblue')
ax1.bar(iters, t_sub, bottom=t_master, label='subproblem', color='orange')
ax1.set_ylabel('Time per iteration (s)')
ax1.set_title('C&CG Gamma=3: time per iteration')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(iters, gap, marker='o', color='red')
ax2.set_ylabel('Outer gap (%)')
ax2.set_xlabel('Iteration')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Check for duplicate scenarios in CCG Gamma=3
r = [r for r in results if r['Gamma'] == 3][0]
log = r['_ccg_log']

J  = list(range(r['J']))
Hn = 2
H  = list(range(Hn))

# Convert each eps_bar to a hashable tuple of the disruption level per facility
# eps_bar[j,h] is binary — find which h is active for each j
def scenario_signature(eps_bar):
    return tuple(
        next(h for h in H if eps_bar[j, h] > 0.5)
        for j in J
    )

signatures = [scenario_signature(d['eps_bar']) for d in log]

print("Scenario per iteration:")
for i, sig in enumerate(signatures):
    print(f"  iter {i+1:2d}: {sig}")

n_unique = len(set(signatures))
print(f"\nUnique scenarios: {n_unique} / {len(signatures)} iterations")
print(f"Duplicates: {len(signatures) - n_unique}")

# Use a matching instance from the main verification grid